# RAG Pipeline — Step by Step Walkthrough

Run each cell one at a time. Each step shows what's happening inside the pipeline.

```
Step 1 → Load Data
Step 2 → Preprocess
Step 3 → Chunking
Step 4 → Embedding
Step 5 → Vector DB (store)
Step 6 → Query Classification
Step 7 → Retrieval
Step 8 → Reranking
Step 9 → Answer Generation  ← needs Ollama running
Step 10 → Evaluation        ← needs Ollama running
```

> Steps 1-8 work without Ollama. Steps 9-10 need `ollama serve` running with `llama3.1:8b`.

In [1]:
# ── Setup: make src/ importable from this notebook ──────────────────────────
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))
print('Python path set. Ready.')

Python path set. Ready.


---
## Step 1 — Load Data
Load one row from the finance dataset (finqa) in RAGBench.
Each row has: `question`, `documents` (list of context strings), `answer`.

In [2]:
from datasets import load_dataset

print('Loading finqa from RAGBench (test split)...')
ds = load_dataset('rungalileo/ragbench', 'finqa', split='test')

# Pick one example row
row = ds[0]

print(f'Total rows in dataset: {len(ds)}')
print(f'\nColumns available: {ds.column_names}')
print(f'\n{"-"*50}')
print(f'QUESTION:\n{row["question"]}')
print(f'\nGOLD ANSWER:\n{row["response"]}')
print(f'\nNumber of context documents in this row: {len(row["documents"])}')
print(f'\nFirst context document (first 300 chars):')
print(row['documents'][0][:300])

Loading finqa from RAGBench (test split)...


Total rows in dataset: 2294

Columns available: ['id', 'question', 'documents', 'response', 'generation_model_name', 'annotating_model_name', 'dataset_name', 'documents_sentences', 'response_sentences', 'sentence_support_information', 'unsupported_response_sentence_keys', 'adherence_score', 'overall_supported_explanation', 'relevance_explanation', 'all_relevant_sentence_keys', 'all_utilized_sentence_keys', 'trulens_groundedness', 'trulens_context_relevance', 'ragas_faithfulness', 'ragas_context_relevance', 'gpt3_adherence', 'gpt3_context_relevance', 'gpt35_utilization', 'relevance_score', 'utilization_score', 'completeness_score']

--------------------------------------------------
QUESTION:
what is the rate of return in cadence design systems inc . of an investment from 2010 to 2011?

GOLD ANSWER:
The rate of return in Cadence Design Systems Inc. from 2010 to 2011 is 37.9%. This is calculated by taking the value on 1/1/2011 (137.90) and subtracting the initial investment value on 1/2/

---
## Step 2 — Preprocess
Clean the raw document text before chunking (removes excess newlines).

In [3]:
from ingestion.preprocess_text import preprocess_documents

raw_doc = row['documents'][0]
cleaned_doc = preprocess_documents(raw_doc)

print(f'Original length : {len(raw_doc)} characters')
print(f'Cleaned length  : {len(cleaned_doc)} characters')
print(f'\nCleaned text (first 400 chars):')
print(cleaned_doc[:400])

Original length : 1092 characters
Cleaned length  : 1092 characters

Cleaned text (first 400 chars):
stockholder return performance graph the following graph compares the cumulative 5-year total stockholder return on our common stock relative to the cumulative total return of the nasdaq composite index and the s&p 400 information technology index . the graph assumes that the value of the investment in our common stock on january 2 , 2010 and in each index on december 31 , 2009 ( including reinves


---
## Step 3 — Chunking
Split the cleaned document into smaller overlapping chunks.

**Why?** LLMs and vector DBs work best with smaller pieces of text (~500-1000 chars), not entire documents.

In [4]:
from chunking.text_chunking import chunk_documents

chunks = chunk_documents(cleaned_doc, chunk_size=1000, chunk_overlap=100)

print(f'Document split into {len(chunks)} chunks')
print(f'Chunk size setting : 1000 chars | Overlap : 100 chars')
print()

for i, chunk in enumerate(chunks):
    print(f'--- Chunk {i+1} ({len(chunk)} chars) ---')
    print(chunk[:250])
    if len(chunk) > 250:
        print('...')
    print()

Document split into 2 chunks
Chunk size setting : 1000 chars | Overlap : 100 chars

--- Chunk 1 (999 chars) ---
stockholder return performance graph the following graph compares the cumulative 5-year total stockholder return on our common stock relative to the cumulative total return of the nasdaq composite index and the s&p 400 information technology index . 
...

--- Chunk 2 (190 chars) ---
12/31/09 in index , including reinvestment of dividends . indexes calculated on month-end basis . copyright a9 2014 s&p , a division of the mcgraw-hill companies inc . all rights reserved. .



---
## Step 4 — Embedding
Convert each chunk into a vector (a list of numbers).

**Why?** Vectors let us do math-based similarity search — similar texts have similar vectors.

In [5]:
from embeddings.embedder import get_embedder
import numpy as np

print('Loading embedding model (BAAI/bge-small-en-v1.5)...')
embedder = get_embedder()

# Embed the first chunk
sample_chunk = chunks[0]
vector = embedder.encode(sample_chunk)

print(f'\nChunk text (first 100 chars): {sample_chunk[:100]}...')
print(f'\nEmbedding vector shape  : {vector.shape}')   # e.g. (384,)
print(f'Vector dimension        : {len(vector)}')
print(f'First 10 values         : {np.round(vector[:10], 4)}')
print(f'Min value               : {vector.min():.4f}')
print(f'Max value               : {vector.max():.4f}')

# Show similarity between two chunks
if len(chunks) >= 2:
    vec2 = embedder.encode(chunks[1])
    similarity = np.dot(vector, vec2) / (np.linalg.norm(vector) * np.linalg.norm(vec2))
    print(f'\nCosine similarity between Chunk 1 and Chunk 2: {similarity:.4f}')
    print('(1.0 = identical, 0.0 = unrelated)')

Loading embedding model (BAAI/bge-small-en-v1.5)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Chunk text (first 100 chars): stockholder return performance graph the following graph compares the cumulative 5-year total stockh...

Embedding vector shape  : (384,)
Vector dimension        : 384
First 10 values         : [-0.0513 -0.0231 -0.0122 -0.0053  0.0221 -0.0049 -0.0257  0.0646  0.0223
  0.012 ]
Min value               : -0.3098
Max value               : 0.3418

Cosine similarity between Chunk 1 and Chunk 2: 0.7668
(1.0 = identical, 0.0 = unrelated)


OPTIONAL - run if only finance needs to be loaded

In [ ]:
# # ── Build finance collection only (run once) ─────────────────────────────────
# from ingestion.build_vectordb import VectorDBBuilder_RAGbench
# from embeddings.embedder import get_embedding_function

# CHROMA_PATH = "../vectordb/chroma_db"

# embedding_fn = get_embedding_function()

# finance_db = VectorDBBuilder_RAGbench(
#     collection_name="finance",
#     embedding_function=embedding_fn,
#     path=CHROMA_PATH
# )

# finance_db.initialize()   # skips ingestion if already built
# finance_db.summary()


---
## Step 5 — Vector DB (Store)


OPTION 1 - for complete data based implementation - USE complete data from chroma DB : 

In [6]:
import chromadb

CHROMA_PATH = "../vectordb/chroma_db"

client = chromadb.PersistentClient(path=CHROMA_PATH)

# Load without embedding_function — collection already has embeddings stored
collection = client.get_collection(name="finance")

print(f"Connected to persistent ChromaDB at: {CHROMA_PATH}")
print(f"Total chunks in finance collection : {collection.count()}")


Connected to persistent ChromaDB at: ../vectordb/chroma_db
Total chunks in finance collection : 3518


Option - 2 :
Store a small set of chunks into ChromaDB so we can search them.

We use an **in-memory** ChromaDB here (not persistent) so we can test quickly without polluting the real DB.

In [ ]:
# import chromadb
# from chromadb import EmbeddingFunction, Embeddings

# # Wrap the already-loaded embedder from Step 4 — no model re-download
# class ReuseEmbedder(EmbeddingFunction):
#     def __call__(self, input):
#         return embedder.encode(input).tolist()

# client = chromadb.EphemeralClient()
# collection = client.get_or_create_collection(
#     name='finance_demo',
#     embedding_function=ReuseEmbedder()
# )

# # Use chunks already created in Step 3 (from 1 document only — keeps it fast)
# ids   = [f'chunk_{i}' for i in range(len(chunks))]
# texts = chunks

# collection.add(documents=texts, ids=ids)

# print(f'Stored {collection.count()} chunks in ChromaDB')
# print(f'\nChunk IDs stored:')
# for cid in ids:
#     print(f'  {cid}')


C:\Users\az845\AppData\Local\Temp\ipykernel_17504\1289491506.py:12: DeprecationWarning: The class ReuseEmbedder does not implement __init__. This will be required in a future version.
  embedding_function=ReuseEmbedder()


Stored 2 chunks in ChromaDB

Chunk IDs stored:
  chunk_0
  chunk_1


look  into the chunks

In [7]:
# Peek inside the ChromaDB collection
print(f'Total chunks stored: {collection.count()}\n')

# Fetch all stored chunks
stored = collection.get()

for i, (chunk_id, text) in enumerate(zip(stored['ids'], stored['documents'])):
    print(f'{"="*60}')
    print(f'ID    : {chunk_id}')
    print(f'Text  : {text}')
    print()


Total chunks stored: 3518

ID    : finance_finqa_0
Text  : stockholder return performance graph the following graph compares the cumulative 5-year total stockholder return on our common stock relative to the cumulative total return of the nasdaq composite index and the s&p 400 information technology index . the graph assumes that the value of the investment in our common stock on january 2 , 2010 and in each index on december 31 , 2009 ( including reinvestment of dividends ) was $ 100 and tracks it each year thereafter on the last day of cadence 2019s fiscal year through january 3 , 2015 and , for each index , on the last day of the calendar comparison of 5 year cumulative total return* among cadence design systems , inc. , the nasdaq composite index , and s&p 400 information technology cadence design systems , inc . nasdaq composite s&p 400 information technology 12/28/13 1/3/151/1/11 12/31/11 12/29/121/2/10 *$ 100 invested on 1/2/10 in stock or 12/31/09 in index , including reinvestm

---
## Step 6 — Query Classification
Before retrieving, the pipeline classifies the query to decide:
- Does this question need RAG (retrieval)?
- Or can the LLM answer it directly from its own knowledge?

> **Needs Ollama running.** Skip to Step 7 if you don't have it yet.

In [11]:
# ── Uncomment this cell when Ollama is running ────────────────────────────────

from classification.query_classifier import build_classifier
from classification.query_strategy import determine_strategy
from llm.model import llm

query = row['question']
print(f'Query: {query}')

classifier = build_classifier(llm=llm)
classification = classifier.invoke({'query': query})

print(f'\nClassification result:')
print(f'  Domain             : {classification["domain"]}')
print(f'  Dataset            : {classification["dataset"]}')
print(f'  Retrieval required : {classification["retrieval_required"]}')
print(f'  Reasoning          : {classification["reasoning"]}')

strategy = determine_strategy(classification)
print(f'\nStrategy chosen: {strategy}')

# print('Cell is commented out — uncomment when Ollama is running.')

Query: what is the rate of return in cadence design systems inc . of an investment from 2010 to 2011?


KeyboardInterrupt: 

---
## Step 7 — Retrieval
Search the vector DB for chunks most similar to the query.

ChromaDB computes cosine similarity between the query embedding and all stored chunk embeddings.

In [8]:
query = row['question']
TOP_K = 5

print(f'Query: {query}')
print(f'Retrieving top {TOP_K} chunks...\n')

# Manually embed the query using the embedder from Step 4
query_vector = embedder.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_vector],
    n_results=TOP_K
)

retrieved_docs  = results['documents'][0]
retrieved_ids   = results['ids'][0]
retrieved_dists = results['distances'][0]

print(f'Retrieved {len(retrieved_docs)} chunks:\n')
for i, (doc_id, dist, doc) in enumerate(zip(retrieved_ids, retrieved_dists, retrieved_docs)):
    print(f'[{i+1}] ID: {doc_id} | Distance: {dist:.4f}')
    print(f'    {doc[:200]}...')
    print()


Query: what is the rate of return in cadence design systems inc . of an investment from 2010 to 2011?
Retrieving top 5 chunks...

Retrieved 5 chunks:

[1] ID: finance_finqa_2 | Distance: 0.2557
    [["", "1/2/2010", "1/1/2011", "12/31/2011", "12/29/2012", "12/28/2013", "1/3/2015"], ["cadence design systems inc .", "100.00", "137.90", "173.62", "224.37", "232.55", "314.36"], ["nasdaq composite", ...

[2] ID: finance_finqa_0 | Distance: 0.2560
    stockholder return performance graph the following graph compares the cumulative 5-year total stockholder return on our common stock relative to the cumulative total return of the nasdaq composite ind...

[3] ID: finance_finqa_1096 | Distance: 0.3230
    performance graph the table below compares the cumulative total shareholder return on our common stock with the cumulative total return of ( i ) the standard & poor's 500 composite stock index ( "s&p ...

[4] ID: finance_finqa_277 | Distance: 0.3273
    financing activities the decrease in cash

---
## Step 8 — Reranking
The initial retrieval uses vector similarity (fast but approximate).
Reranking uses a **cross-encoder** model that reads the query + each doc together for a more accurate relevance score.

The cross-encoder is slower but much more precise — it's a second-pass filter.

In [9]:
from reranking.reranker import rerank

TOP_K_RERANK = 3

print(f'Reranking {len(retrieved_docs)} retrieved chunks → keeping top {TOP_K_RERANK}\n')

reranked_docs = rerank(query, results)[:TOP_K_RERANK]

print(f'\nTop {TOP_K_RERANK} docs after reranking:')
for i, doc in enumerate(reranked_docs):
    print(f'\n[{i+1}] {doc[:300]}...')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Reranking 5 retrieved chunks → keeping top 3

2026-06-21 01:17:49,730 | INFO | reranking.reranker | --------------------------------------------------
2026-06-21 01:17:49,731 | INFO | reranking.reranker | Reranking retrieved documents using cross-encoder model...
2026-06-21 01:17:49,731 | INFO | reranking.reranker | Number of retrieved documents: 5. Returning top 5 reranked documents.
2026-06-21 01:17:51,823 | INFO | reranking.reranker | Score: 0.8107, Doc ID: finance_finqa_2, Doc: [["", "1/2/2010", "1/1/2011", "12/31/2011", "12/29...
2026-06-21 01:17:51,824 | INFO | reranking.reranker | Score: 0.2082, Doc ID: finance_finqa_0, Doc: stockholder return performance graph the following...
2026-06-21 01:17:51,824 | INFO | reranking.reranker | Score: 0.0083, Doc ID: finance_finqa_435, Doc: [["millions", "2011", "2010", "2009"], ["cash prov...
2026-06-21 01:17:51,824 | INFO | reranking.reranker | Score: 0.0006, Doc ID: finance_finqa_1096, Doc: performance graph the table below compares the cu

DEBUGGER TO SEE WHAT WAS RETRIEVED:

In [11]:
# Confirm Fix 2 worked — chunks should now be readable text, not raw JSON
print("Sample retrieved chunk after Fix 2:")
print(reranked_docs[0])


Sample retrieved chunk after Fix 2:
[["", "1/2/2010", "1/1/2011", "12/31/2011", "12/29/2012", "12/28/2013", "1/3/2015"], ["cadence design systems inc .", "100.00", "137.90", "173.62", "224.37", "232.55", "314.36"], ["nasdaq composite", "100.00", "117.61", "118.70", "139.00", "196.83", "223.74"], ["s&p 400 information technology", "100.00", "128.72", "115.22", "135.29", "173.25", "187.84"]]


---
## Step 9 — Answer Generation
Send the query + top reranked chunks to the LLM to generate an answer.

> **Needs Ollama running**: `ollama serve` + `ollama pull llama3.1:8b`

In [14]:
# ── Uncomment when Ollama is running ─────────────────────────────────────────

from llm.answer_generation import generate_answer
from prompts.answer_prompt import ANSWER_PROMPT

print(f'Query: {query}\n')
print('Prompt sent to LLM (preview):')
print(ANSWER_PROMPT.format(query=query, context=reranked_docs[:2])[:600])
print('...')
print('\nGenerating answer...')

answer = generate_answer(query, reranked_docs)
print(f'\nGENERATED ANSWER:\n{answer}')
print(f'\nGOLD ANSWER:\n{row["response"]}')

print('Cell is commented out — uncomment when Ollama is running.')

Query: what is the rate of return in cadence design systems inc . of an investment from 2010 to 2011?

Prompt sent to LLM (preview):

You are a helpful RAG assistant.
Answer ONLY from the provided context.
If the answer is not present,
say:
"I couldn't find enough evidence."

Context:
['[["", "1/2/2010", "1/1/2011", "12/31/2011", "12/29/2012", "12/28/2013", "1/3/2015"], ["cadence design systems inc .", "100.00", "137.90", "173.62", "224.37", "232.55", "314.36"], ["nasdaq composite", "100.00", "117.61", "118.70", "139.00", "196.83", "223.74"], ["s&p 400 information technology", "100.00", "128.72", "115.22", "135.29", "173.25", "187.84"]]', 'stockholder return performance graph the following graph compares the cumulative 5-ye
...

Generating answer...
2026-06-21 00:29:30,752 | INFO | llm.answer_generation | Generating answer using LLM...
2026-06-21 00:30:13,952 | INFO | llm.answer_generation | Answer generated successfully.
2026-06-21 00:30:13,953 | INFO | llm.answer_generation | -------

---
## Step 10 — Evaluation
Score the generated answer using an LLM-as-judge that computes the 4 RAGBench metrics.

| Metric | What it measures |
|---|---|
| Context Relevance | Were the retrieved chunks relevant to the question? |
| Context Utilization | Did the LLM actually use what was retrieved? |
| Completeness | Did the answer cover all relevant facts? |
| Adherence | Is every claim in the answer backed by the context? |

> **Needs Ollama running**

In [19]:
# ── Uncomment when Ollama is running ─────────────────────────────────────────

from evaluation.evaluation_metrics import evaluate_rag_response, calculate_metrics

eval_result = evaluate_rag_response(
    question=query,
    contexts=reranked_docs,
    answer=answer
)

metrics = calculate_metrics(eval_result)

print('RAG Evaluation Results:')
print(f'  Context Relevance    : {metrics["context_relevance"]:.2f}')
print(f'  Context Utilization  : {metrics["context_utilization"]:.2f}')
print(f'  Completeness         : {metrics["completeness"]:.2f}')
print(f'  Adherence            : {metrics["adherence"]:.2f}')
print(f'\nJudge reasoning: {eval_result.reasoning}')

# print('Cell is commented out — uncomment when Ollama is running.')

RAG Evaluation Results:
  Context Relevance    : 0.50
  Context Utilization  : 0.14
  Completeness         : 0.33
  Adherence            : 0.75

Judge reasoning: The answer is mostly supported by the retrieved context. The initial and final values for Cadence Design Systems Inc. in 2010 and 2011 are correctly identified as $100.00 and $137.90 respectively. However, the calculation of percentage change could be more precise with additional context facts such as the exact dates or a clear indication that dividends were reinvested.


Improved prompt for better retrieval:

In [18]:
import importlib
import prompts.answer_prompt as ap
import llm.answer_generation as ag

# Reload both — order matters
importlib.reload(ap)
importlib.reload(ag)

answer = ag.generate_answer(query, reranked_docs)
print(f'GENERATED ANSWER:\n{answer}')
print(f'\nGOLD ANSWER:\n{row["response"]}')



2026-06-21 00:55:02,915 | INFO | llm.answer_generation | Generating answer using LLM...
2026-06-21 00:56:21,404 | INFO | llm.answer_generation | Answer generated successfully.
2026-06-21 00:56:21,405 | INFO | llm.answer_generation | --------------------------------------------------
GENERATED ANSWER:
To find the rate of return, we need to calculate the percentage change in value from 2010 to 2011.

From the context, we have:

* The initial value on January 2, 2010: $100.00
* The final value on January 1, 2011: $137.90

First, let's find the difference:
$137.90 - $100.00 = $37.90

Next, let's calculate the percentage change:
($37.90 ÷ $100.00) x 100% ≈ 37.9%

So, the rate of return in Cadence Design Systems Inc. from 2010 to 2011 is approximately 37.9%.

GOLD ANSWER:
The rate of return in Cadence Design Systems Inc. from 2010 to 2011 is 37.9%. This is calculated by taking the value on 1/1/2011 (137.90) and subtracting the initial investment value on 1/2/2010 (100.00).


RUN THE BELOW ONE TIME SCRIPT - to fix the data stored in vector db as json embedded... with this it would be properly stored as structured embedded

In [13]:
# ── Delete and rebuild finance collection ─────────────────────────────────────
import chromadb
from ingestion.build_vectordb import VectorDBBuilder_RAGbench
from embeddings.embedder import get_embedding_function

CHROMA_PATH = "../vectordb/chroma_db"

# Step 1: Delete existing finance collection
client = chromadb.PersistentClient(path=CHROMA_PATH)
client.delete_collection(name="finance")
print("Finance collection deleted.")

# Step 2: Rebuild with updated preprocess_text.py (tables → readable text)
embedding_fn = get_embedding_function()
finance_db = VectorDBBuilder_RAGbench(
    collection_name="finance",
    embedding_function=embedding_fn,
    path=CHROMA_PATH
)
finance_db.initialize()
finance_db.summary()


Finance collection deleted.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Collection 'finance' is empty. Starting ingestion...
2026-06-21 01:22:17,562 | INFO | ingestion.build_vectordb | Loading Dataset: finqa
2026-06-21 01:22:20,496 | INFO | ingestion.build_vectordb | Total records in finqa are 2294. Deduped docs are 1097
Document 1 split into 2 chunks.
Document 2 split into 1 chunks.
Document 3 split into 1 chunks.
Document 4 split into 5 chunks.
Document 5 split into 1 chunks.
Document 6 split into 3 chunks.
Document 7 split into 1 chunks.
Document 8 split into 2 chunks.
Document 9 split into 2 chunks.
Document 10 split into 1 chunks.
Document 11 split into 1 chunks.
Document 12 split into 3 chunks.
Document 13 split into 2 chunks.
Document 14 split into 2 chunks.
Document 15 split into 3 chunks.
Document 16 split into 1 chunks.
Document 17 split into 1 chunks.
Document 18 split into 1 chunks.
Document 19 split into 1 chunks.
Document 20 split into 1 chunks.
Document 21 split into 2 chunks.
Document 22 split into 1 chunks.
Document 23 split into 5 chunks.

ValueError: Non-empty lists are required for ['ids', 'metadatas', 'documents'] in add.